# Modelo Híbrido: ODE + Data Science — Pilas SAG vs. Teniente 8
## División El Teniente — Codelco | 2026

---

### Skills aplicados
- `skill_molienda_sag` — proceso físico del sistema
- `skill_series_temporales_industriales` — análisis temporal
- `skill_machine_learning_operacional` — RF, XGBoost, SHAP
- `skill_estadistica_bayesiana_avanzada` — Monte Carlo
- `skill_explainable_ai_governance` — explicabilidad causal
- `skill_data_scientist_senior` — metodología integral
- `skill_product_owner_analitica_minera` — decisión operacional
- `skill_forecasting_industrial` — simulación de escenarios

---

### Estructura del modelo

| Fase | Contenido |
|------|-----------|
| 1 | Calibración ODE: inventario observado vs. modelado |
| 2 | Regresiones: TPH ~ Pila + T8 + Correa + Config |
| 3 | Detección de umbrales: LOWESS + Árbol de decisión |
| 4 | Análisis de sensibilidad: Tornado + Heatmaps |
| 5 | Monte Carlo: 10,000 simulaciones, P(agotamiento) |
| 6 | ML + SHAP: RF, XGBoost, importancia causal |
| 7 | Curvas estratégicas: Pila→TPH, T8→Autonomía |
| 8 | Superficies 3D de respuesta |
| 9 | Mapa operacional: semáforo SAG1 × SAG2 |

---

### Ecuaciones del modelo

$$\frac{dS_i}{dt} = Q_{\text{in},i}(t) - Q_{\text{out},i}(t)$$

$$Q_{\text{out},i} = \frac{\text{TPH}_{\text{SAG},i}}{\text{Cap}_i} \times 100 \; [\%/h]$$

**Calibración:** $Q_{\text{in,total}} = Q_{\text{T8}} + Q_{\text{otras fuentes}}$
(donde $Q_{\text{otras fuentes}}$ se estima del estado de equilibrio no-T8)

In [ ]:
import subprocess, sys
from pathlib import Path

BASE = Path('C:/Users/jorel038/OneDrive - Codelco/Documentos/AA_CIO_DET/07_Rendimientos')

print('Ejecutando Modelo Hibrido ODE + Data Science...')
r = subprocess.run(
    [sys.executable, '-X', 'utf8', str(BASE / 'src/modelo_hibrido.py')],
    capture_output=True, text=True, encoding='utf-8'
)
print(r.stdout)
if r.returncode != 0:
    print('STDERR:', r.stderr[-3000:])

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

FIG_DIR = BASE / 'outputs/figures/modelo_hibrido'

figuras = [
    ('F1_calibracion_ode.png',        'Fase 1 — Calibracion ODE: Inventario Observado vs Modelado'),
    ('F2_regresiones_sag1.png',       'Fase 2 — Regresiones SAG1: TPH ~ Pila + T8 + Correa + Config'),
    ('F2_regresiones_sag2.png',       'Fase 2 — Regresiones SAG2'),
    ('F3_umbrales_lowess_dt.png',     'Fase 3 — Umbrales: LOWESS + Arbol de Decision'),
    ('F4_tornado_sensibilidad.png',   'Fase 4 — Tornado: Sensibilidad OAT'),
    ('F4_heatmap_nivel_duracion.png', 'Fase 4 — Heatmap: Nivel Inicial x Duracion -> Nivel Final'),
    ('F4_heatmap_nivel_rate.png',     'Fase 4 — Heatmap: Nivel x Rate -> Autonomia'),
    ('F5_montecarlo_sag1.png',        'Fase 5 — Monte Carlo SAG1: 10,000 simulaciones'),
    ('F5_montecarlo_sag2.png',        'Fase 5 — Monte Carlo SAG2'),
    ('F6_ml_shap_sag1.png',           'Fase 6 — ML + SHAP SAG1: RF vs XGBoost'),
    ('F6_shap_dependence_sag1.png',   'Fase 6 — SHAP Dependence SAG1'),
    ('F6_ml_shap_sag2.png',           'Fase 6 — ML + SHAP SAG2'),
    ('F6_shap_dependence_sag2.png',   'Fase 6 — SHAP Dependence SAG2'),
    ('F7_curvas_estrategicas.png',    'Fase 7 — Curvas Estrategicas: Causa x Efecto x Riesgo'),
    ('F8_superficies_3d.png',         'Fase 8 — Superficies de Respuesta 3D'),
    ('F9_mapa_operacional.png',       'Fase 9 — Mapa Operacional: Semaforo SAG1 x SAG2'),
]

for fname, title in figuras:
    fpath = FIG_DIR / fname
    if not fpath.exists():
        print(f'No encontrado: {fname}')
        continue
    fig, ax = plt.subplots(figsize=(16, 8))
    ax.imshow(mpimg.imread(str(fpath)))
    ax.axis('off')
    ax.set_title(title, fontsize=10, pad=6)
    plt.tight_layout()
    plt.show()

In [ ]:
import pandas as pd

XLS = BASE / 'outputs/excel/modelo_hibrido_resultados.xlsx'

print('=== CALIBRACION ODE ===')
print(pd.read_excel(XLS, sheet_name='ODE_Calibracion').to_string(index=False))

print('\n=== REGRESIONES (R2 por modelo) ===')
print(pd.read_excel(XLS, sheet_name='Regresiones').to_string(index=False))

print('\n=== MONTE CARLO (probabilidades agotamiento) ===')
print(pd.read_excel(XLS, sheet_name='MonteCarlo').to_string(index=False))

print('\n=== ML PERFORMANCE ===')
print(pd.read_excel(XLS, sheet_name='ML_Performance').to_string(index=False))

print('\n=== SENSIBILIDAD (muestra: S0=60%, dur=8h) ===')
df_s = pd.read_excel(XLS, sheet_name='Sensibilidad')
mask = (df_s['S0_%'] == 60) & (df_s['dur_h'] == 8)
print(df_s[mask][['SAG','percentil','rate_%_h','S_end_%','autonomia_h','agotado_20']].to_string(index=False))

## Notas de interpretacion

### Calibracion ODE (Fase 1)
- RMSE alto (43-53%) porque CV315/CV316 son **alimentacion parcial** de las pilas (solo contribucion T8).
- El R2 negativo indica que el modelo ODE con solo correas T8 es peor que un simple promedio → refleja que otras fuentes de alimentacion no estan capturadas.
- **Interpretacion correcta:** el modelo captura el efecto de T8 sobre el delta de la pila, no el nivel absoluto.

### Regresiones (Fase 2)
- R2 bajos (0.002–0.075) indican alta variabilidad inherente del proceso.
- El **nivel de pila** no es el principal driver del TPH en 5-min: la molienda SAG opera a tasa constante mientras haya material disponible (arriba de umbrales).
- El efecto T8 es estadisticamente significativo (p < 0.05 en coeficiente `en_t8_int`).

### Monte Carlo (Fase 5)
- SAG2 tiene **mayor riesgo de agotamiento** que SAG1 dado el nivel inicial historico tipico.
- Las probabilidades son altas (38-66% para <20%) porque la distribucion historica de S0 pre-T8 incluye niveles bajos.

### ML/SHAP (Fase 6)
- R2 negativos son esperados: el TPH en 5-min es dificil de predecir con 1h de lag (alta ruido, comportamiento de proceso quasi-discontinuo).
- Los **valores SHAP son validos para importancia relativa** aunque el modelo no prediga bien en absoluto.
- Usar SHAP para ranking de factores, no para prediccion puntual.